# Runtime Resolution Tradeoff Audit

Ce notebook reprend le script `runtime_resolution_tradeoff_audit.py`.

But : rendre l'experience lisible et executable dans Jupyter, avec des explications simples en francais.

Utilisation : executez les cellules dans l'ordre. La derniere cellule lance le script avec des arguments controles.


## Pertinence pour le live streaming

Decision : garde. Teste resolution pose vs vitesse/stabilite pour atteindre le live.

Regle appliquee : l'experience doit aider a entrainer, choisir, calibrer, tester ou executer une prediction en flux video avec seulement les informations disponibles a l'instant courant.


## Pourquoi ce notebook est garde

- Description du script : Audit pose-resolution runtime and score tradeoffs.
- Commande de reproduction referencee : pose-resolution runtime tradeoff audit.
- Artefacts controles : Pose-resolution runtime tradeoff audit exists. (`runs/exp_059_runtime_resolution_tradeoff/metrics/runtime_resolution_tradeoff.csv`).
- Run par defaut : `runs/exp_059_runtime_resolution_tradeoff`.

Decision : garde, car il correspond a un artefact experimental, une commande de reproduction, ou un audit du catalogue.


## Avant de commencer

- Verifiez que les donnees et les dossiers `runs/` attendus existent.
- Le notebook n'a pas ete execute pendant sa creation.
- Les cellules de lancement creent un nom ou un dossier unique quand cela evite d'ecraser un resultat existant.


In [ ]:
# Compatibilite Jupyter
# Certains scripts utilisent __file__. Dans un notebook, on le definit explicitement.
from pathlib import Path

PROJECT_ROOT = Path.cwd()
__file__ = str(PROJECT_ROOT / "runtime_resolution_tradeoff_audit.py")


## Importations et configuration

Cette cellule charge les bibliotheques et definit les constantes utilisees par le script.

In [ ]:
import argparse
from pathlib import Path

import numpy as np
import pandas as pd

from ml_pipeline import ROOT, write_json
from sequence_experiments import append_report, make_run_dir


RUNS = [
    ("640_cuda", "runs/exp_053_realtime_variant_benchmark", 640, "cuda_default"),
    ("576_cuda", "runs/exp_058_realtime_variant_benchmark_imgsz576", 576, "cuda_default"),
    ("512_cuda", "runs/exp_056_realtime_variant_benchmark_imgsz512", 512, "cuda_default"),
    ("384_cuda", "runs/exp_057_realtime_variant_benchmark_imgsz384", 384, "cuda_default"),
    ("640_cpu_light", "runs/exp_055_realtime_variant_benchmark_cpu_light", 640, "cpu_light"),
]


## Fonction `resolve`

Cette cellule definit `resolve`. Elle prepare une partie du script.

In [ ]:
def resolve(path):
    path = Path(path)
    if not path.is_absolute():
        path = ROOT / path
    return path


## Fonction `read_run`

Cette cellule definit `read_run`. Elle prepare une partie du script.

In [ ]:
def read_run(run_path):
    run_path = resolve(run_path)
    summary_path = run_path / "runtime_variant_total_summary.csv"
    pred_path = run_path / "runtime_variant_predictions.csv"
    if not summary_path.exists() or not pred_path.exists():
        return pd.DataFrame(), pd.DataFrame()
    return pd.read_csv(summary_path), pd.read_csv(pred_path)


## Fonction `compare_predictions`

Cette cellule definit `compare_predictions`. Elle prepare une partie du script.

In [ ]:
def compare_predictions(reference, candidate):
    rows = []
    if reference.empty or candidate.empty:
        return rows
    for variant in sorted(set(reference["variant"]) & set(candidate["variant"])):
        ref_v = reference[reference["variant"].eq(variant)].copy()
        cand_v = candidate[candidate["variant"].eq(variant)].copy()
        if ref_v.empty or cand_v.empty:
            continue
        score_col = str(cand_v["score_col"].iloc[0])
        if score_col not in ref_v.columns or score_col not in cand_v.columns:
            continue
        merged = ref_v[["frame", score_col, "alarm"]].merge(
            cand_v[["frame", score_col, "alarm"]],
            on="frame",
            suffixes=("_reference", "_candidate"),
        )
        if merged.empty:
            continue
        diff = (merged[f"{score_col}_reference"] - merged[f"{score_col}_candidate"]).abs()
        alarm_diff = merged["alarm_reference"].astype(int).ne(merged["alarm_candidate"].astype(int))
        rows.append(
            {
                "variant": variant,
                "score_col": score_col,
                "rows_compared": int(len(merged)),
                "score_max_abs_diff_vs_640": float(diff.max()),
                "score_mean_abs_diff_vs_640": float(diff.mean()),
                "score_p95_abs_diff_vs_640": float(np.percentile(diff, 95)),
                "alarm_diff_frames_vs_640": int(alarm_diff.sum()),
            }
        )
    return rows


## Fonction `first_alarm`

Cette cellule definit `first_alarm`. Elle prepare une partie du script.

In [ ]:
def first_alarm(pred, variant):
    group = pred[pred["variant"].eq(variant)]
    if group.empty:
        return np.nan
    alarms = group[group["alarm"].astype(int).eq(1)]
    return np.nan if alarms.empty else float(alarms["time_s"].iloc[0])


## Fonction `max_score`

Cette cellule definit `max_score`. Elle prepare une partie du script.

In [ ]:
def max_score(pred, variant):
    group = pred[pred["variant"].eq(variant)]
    if group.empty:
        return np.nan
    score_col = str(group["score_col"].iloc[0])
    return float(pd.to_numeric(group[score_col], errors="coerce").max())


## Fonction `write_summary`

Cette cellule definit `write_summary`. Elle prepare une partie du script.

In [ ]:
def write_summary(run_dir, rows):
    df = pd.DataFrame(rows)
    lines = ["# Runtime Resolution Tradeoff Audit", ""]
    lines.append("This audit tests whether reducing YOLO pose input resolution can make the causal runtime variants fast enough while preserving the 640-pixel risk trace.")
    lines.append("")
    lines.append("## Summary")
    lines.append("")
    lines.append("| run | variant | imgsz | backend | steady FPS | steady mean ms | steady p95 ms | first alarm s | score p95 diff vs 640 | alarm diff frames | pass mean 30 FPS | pass p95 30 FPS |")
    lines.append("|---|---|---:|---|---:|---:|---:|---:|---:|---:|---:|---:|")
    for _, row in df.sort_values(["variant", "imgsz"], ascending=[True, False]).iterrows():
        lines.append(
            f"| {row['run_label']} | {row['variant']} | {int(row['imgsz'])} | {row['backend']} | "
            f"{row['estimated_fps_from_steady_mean']:.2f} | {row['steady_mean_ms']:.2f} | {row['steady_p95_ms']:.2f} | "
            f"{row['first_alarm_time_s'] if pd.notna(row['first_alarm_time_s']) else 'NA'} | "
            f"{row['score_p95_abs_diff_vs_640'] if pd.notna(row['score_p95_abs_diff_vs_640']) else 'NA'} | "
            f"{int(row['alarm_diff_frames_vs_640']) if pd.notna(row['alarm_diff_frames_vs_640']) else 'NA'} | "
            f"{row['pass_mean_30fps']} | {row['pass_p95_30fps']} |"
        )
    lines.append("")
    lines.append("## Interpretation")
    lines.append("")
    lines.append("- `imgsz=576` improves mean FPS for the fastest fused variant above 30 FPS, but p95 latency remains above a 33.4 ms frame budget.")
    lines.append("- Lower pose resolutions also shift the risk trace versus the 640 baseline. This is useful engineering evidence, not enough to claim final fused production runtime.")
    lines.append("- The conservative deployment conclusion remains: core sequence-only danger has a real-time candidate; final fused trajectory+attention+PPE still needs optimization and validation before production real-time claims.")
    summary_path = run_dir / "runtime_resolution_tradeoff_summary.md"
    summary_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    append_report(run_dir, "Runtime Resolution Tradeoff Audit", f"- Summary: `{summary_path}`")
    return summary_path


## Fonction `run`

Cette cellule definit `run`. Elle prepare une partie du script.

In [ ]:
def run(args):
    run_dir = make_run_dir(args.run_name)
    reference_summary, reference_pred = read_run(args.reference_run)
    if reference_summary.empty or reference_pred.empty:
        raise SystemExit(f"Reference run is missing required files: {args.reference_run}")
    rows = []
    for label, path, imgsz, backend in RUNS:
        summary, pred = read_run(path)
        if summary.empty or pred.empty:
            continue
        comparison = pd.DataFrame(compare_predictions(reference_pred, pred))
        for _, item in summary.iterrows():
            variant = str(item["variant"])
            comp = comparison[comparison["variant"].eq(variant)]
            comp_row = comp.iloc[0].to_dict() if not comp.empty else {}
            row = {
                "run_label": label,
                "run_path": path,
                "imgsz": int(imgsz),
                "backend": backend,
                "variant": variant,
                "steady_mean_ms": float(item["steady_mean_ms"]),
                "steady_p95_ms": float(item["steady_p95_ms"]),
                "estimated_fps_from_steady_mean": float(item["estimated_fps_from_steady_mean"]),
                "first_alarm_time_s": first_alarm(pred, variant),
                "max_score": max_score(pred, variant),
                "pass_mean_30fps": bool(float(item["estimated_fps_from_steady_mean"]) >= 30.0),
                "pass_p95_30fps": bool(float(item["steady_p95_ms"]) <= 33.4),
                "score_max_abs_diff_vs_640": np.nan,
                "score_mean_abs_diff_vs_640": np.nan,
                "score_p95_abs_diff_vs_640": np.nan,
                "alarm_diff_frames_vs_640": np.nan,
            }
            row.update(comp_row)
            if label == "640_cuda":
                row.update(
                    {
                        "score_max_abs_diff_vs_640": 0.0,
                        "score_mean_abs_diff_vs_640": 0.0,
                        "score_p95_abs_diff_vs_640": 0.0,
                        "alarm_diff_frames_vs_640": 0,
                    }
                )
            rows.append(row)
    out = pd.DataFrame(rows)
    out.to_csv(run_dir / "metrics" / "runtime_resolution_tradeoff.csv", index=False)
    write_json(
        run_dir / "metrics" / "runtime_resolution_tradeoff_config.json",
        {
            "reference_run": args.reference_run,
            "runs": [{"label": label, "path": path, "imgsz": imgsz, "backend": backend} for label, path, imgsz, backend in RUNS],
            "pass_mean_30fps": "estimated steady-state FPS >= 30",
            "pass_p95_30fps": "steady-state p95 latency <= 33.4 ms",
        },
    )
    summary_path = write_summary(run_dir, rows)
    print(run_dir)
    print(summary_path)


## Point d'entree principal

Cette cellule definit `main`. Elle prepare une partie du script.

In [ ]:
def main():
    parser = argparse.ArgumentParser(description="Audit pose-resolution runtime and score tradeoffs.")
    parser.add_argument("--run-name", default="exp_059_runtime_resolution_tradeoff")
    parser.add_argument("--reference-run", default="runs/exp_053_realtime_variant_benchmark")
    args = parser.parse_args()
    run(args)


## Lancer le script

Cette cellule lance le `main()` avec des arguments adaptes au notebook.

In [ ]:
# Lancement du script
# Modifiez NOTEBOOK_ARGS si vous voulez changer les options.
from datetime import datetime
import sys

RUN_NAME_BASE = "exp_059_runtime_resolution_tradeoff_notebook"
RUN_NAME = f"{RUN_NAME_BASE}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
NOTEBOOK_ARGS = ["--run-name", RUN_NAME]

ancien_argv = sys.argv[:]
sys.argv = ["runtime_resolution_tradeoff_audit.py"] + NOTEBOOK_ARGS
try:
    print("Arguments utilises :", sys.argv[1:])
    main()
finally:
    sys.argv = ancien_argv
